# 🛍️ Customer Segmentation via K-Means Clustering & Behavioral Profiling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshamith777-cpu/AIML_3_CustomerSegmentation_byte/blob/main/notebooks/customer_segmentation.ipynb)

**Repository:** `AIML_3_CustomerSegmentation_byte`  
**Track:** Arithmatrix Virtual Internship Program (AVIP 2026) — AI/ML Track (Task 3)  
**Algorithm:** K-Means Clustering (`k-means++` initialization, $n_{\text{init}}=50$, $k=5$)  
**Dataset:** Real Kaggle Mall Customer Segmentation Benchmark (200 shoppers)

---

### ⚠️ Unsupervised Paradigm Shift: No Labels, No Train/Test Split, No Accuracy
* **Unsupervised Nature:** Unlike supervised classification (Tasks 1 & 2), there are **no target labels** ($y$) in this dataset. The algorithm discovers intrinsic geometric structures directly from feature distributions.
* **Why No Train/Test Split?** In unsupervised clustering, partitioning data into training and testing slices is counterproductive for exploratory market segmentation. The business objective is to cluster **100% of available mall shoppers** to reveal complete customer archetypes.
* **Evaluation Criteria:** There is **no accuracy score** in unsupervised learning. Model quality is mathematically evaluated using **Inertia (Within-Cluster Sum of Squares)** and **Silhouette Analysis**, paired with actionable **business interpretability**.

### Step 1: Environment Setup & Deterministic Seed
We import standard data science libraries and pin `random_state=42` to guarantee exact cluster reproducibility across runs.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from mpl_toolkits.mplot3d import Axes3D

RANDOM_STATE = 42

# Ensure output directories exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("Environment initialized with deterministic seed = 42.")

### Step 2: Data Ingestion & Exploratory Data Analysis (EDA)

#### 🛡️ Decision 1: Drop `CustomerID` (Non-Negotiable)
`CustomerID` is an arbitrary sequential database index ($1, 2, 3, \dots$). If left in the feature matrix, K-Means will compute Euclidean distance on index values, artificially grouping customers by their arrival order rather than behavioral patterns. We explicitly drop `CustomerID`.

In [ ]:
# Load authentic Kaggle Mall Customer dataset
df_raw = pd.read_csv('../data/Mall_Customers.csv')
print(f"Raw Dataset Shape: {df_raw.shape}")
print(df_raw.head())

# Documented Decision: Drop CustomerID
df = df_raw.drop(columns=['CustomerID'])
print(f"\nShape after dropping CustomerID: {df.shape}")
print(df.describe().round(2))

# Export dataset sample (Required AVIP deliverable)
df.head(10).to_csv('../outputs/dataset_sample.csv', index=False)
print("\nSaved ../outputs/dataset_sample.csv (10 rows)")

# Generate EDA Pairplot
plt.style.use('dark_background')
g = sns.pairplot(
    df.drop(columns=['Gender']),
    corner=True,
    diag_kind='kde',
    plot_kws={'alpha': 0.7, 'color': '#C8FF32', 'edgecolor': '#141610', 's': 50},
    diag_kws={'color': '#78B7FF', 'fill': True}
)
g.fig.suptitle('EDA Pairplot — Mall Customer Distributions', y=1.02, fontsize=14, color='#F2F0E7', weight='bold')
g.savefig('../outputs/eda_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved ../outputs/eda_pairplot.png")

### Step 3: Feature Selection & Scaling

#### 🛡️ Decision 2: Standardize Continuous Features (Non-Negotiable)
K-Means computes pairwise Euclidean distances:
$$d(x, c) = \sqrt{\sum_{j=1}^p (x_j - c_j)^2}$$
`Annual Income` ranges from $15k to $137k (scale of $\sim 100$), whereas `Age` ranges from 18 to 70 and `Spending Score` from 1 to 99. Without normalization, Annual Income would mathematically dominate cluster formation. We scale all continuous features using `StandardScaler` to zero mean and unit variance.

#### 🛡️ Decision 3: Exclude `Gender` from Distance Calculation
`Gender` is a discrete binary categorical variable. Projecting binary values into Euclidean distance space creates an artificial split plane that distorts spherical cluster geometry. Following best practices, we exclude Gender from feature clustering, but track and report gender breakdown per cluster during business profiling.

In [ ]:
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Feature Means (expect ~0.0):", X_scaled.mean(axis=0).round(4))
print("Scaled Feature Stds  (expect 1.0):", X_scaled.std(axis=0).round(4))

### Step 4: Hyperparameter Optimization — Justifying $k$ (Elbow & Silhouette)

We test cluster candidates $k \in [2, 10]$ using two complementary mathematical validation techniques:
1. **Elbow Method:** Measures total Within-Cluster Sum of Squares (Inertia). An inflection bend indicates diminishing returns in variance reduction.
2. **Silhouette Analysis:** Measures how tightly cohesive points are to their own cluster relative to neighboring clusters:
   $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
   Values close to $+1$ indicate well-separated clusters.

In [ ]:
K_RANGE = range(2, 11)
inertias, silhouettes = [], []

print("--- K Optimization Table ---")
print("k | Inertia    | Silhouette")
print("-" * 35)

for k in K_RANGE:
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=50, max_iter=300, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X_scaled)
    inertias.append(km_k.inertia_)
    sil = silhouette_score(X_scaled, labels_k)
    silhouettes.append(sil)
    marker = " <-- OPTIMAL (k=5)" if k == 5 else ""
    print(f"{k} | {km_k.inertia_:10.2f} | {sil:.4f}{marker}")

# Side-by-Side Dual Confirmation Plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5), facecolor='#141610')
for a in ax:
    a.set_facecolor('#1B1E16')
    a.tick_params(colors='#F2F0E7')
    a.xaxis.label.set_color('#F2F0E7')
    a.yaxis.label.set_color('#F2F0E7')
    a.title.set_color('#F2F0E7')
    for spine in a.spines.values():
        spine.set_color('#333333')

# Plot 1: Elbow Method
ax[0].plot(list(K_RANGE), inertias, marker='o', color='#78B7FF', linewidth=2.5, markersize=7)
ax[0].set_xlabel('Number of Clusters (k)', fontsize=12)
ax[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
ax[0].set_title('Elbow Method (Elbow Bend at k=5)', fontsize=13, weight='bold')
ax[0].axvline(x=5, color='#C8FF32', linestyle='--', linewidth=2, alpha=0.8, label='Optimal k=5')
ax[0].grid(True, linestyle=':', alpha=0.3, color='#F2F0E7')
ax[0].legend(facecolor='#22261C', edgecolor='#333333', labelcolor='#F2F0E7')

# Plot 2: Silhouette Analysis
ax[1].plot(list(K_RANGE), silhouettes, marker='s', color='#C8FF32', linewidth=2.5, markersize=7)
ax[1].set_xlabel('Number of Clusters (k)', fontsize=12)
ax[1].set_ylabel('Silhouette Score', fontsize=12)
ax[1].set_title(f'Silhouette Analysis (Optimal k=5, Score={silhouettes[3]:.4f})', fontsize=13, weight='bold')
ax[1].axvline(x=5, color='#78B7FF', linestyle='--', linewidth=2, alpha=0.8, label='Optimal k=5')
ax[1].grid(True, linestyle=':', alpha=0.3, color='#F2F0E7')
ax[1].legend(facecolor='#22261C', edgecolor='#333333', labelcolor='#F2F0E7')

plt.tight_layout()
plt.savefig('../outputs/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved ../outputs/elbow_silhouette.png (dual confirmation)")

### Step 5: Final K-Means Model Fitting ($k=5$)
We fit K-Means with `init='k-means++'` to avoid suboptimal local minima, setting `n_init=50` to evaluate 50 independent centroid seeds.

In [ ]:
km = KMeans(n_clusters=5, init='k-means++', n_init=50, max_iter=300, random_state=RANDOM_STATE)
df['cluster'] = km.fit_predict(X_scaled)

print("Cluster Assignment Value Counts:")
print(df['cluster'].value_counts().sort_index())

### Step 6: Centroids in REAL UNITS (Inverse Transformation)

#### ⚠️ Critical Failure Prevention:
Raw centroids from `km.cluster_centers_` are in normalized $z$-score coordinates (e.g. $[-1.2, +0.8]$), which are meaningless for marketing strategy. We apply `scaler.inverse_transform` to report centroids in original human-readable physical units (Years of Age, $\$k$ of Annual Income, Points of Spending Score).

In [ ]:
centroids_scaled = km.cluster_centers_
centroids_real = scaler.inverse_transform(centroids_scaled)

cent_df = pd.DataFrame(centroids_real, columns=features).round(2)
cent_df.insert(0, 'cluster', range(5))
cluster_counts = df['cluster'].value_counts().sort_index().values
cent_df.insert(1, 'size', cluster_counts)

cent_df.to_csv('../outputs/centroids.csv', index=False)
print("Centroids in Real Physical Units (outputs/centroids.csv):")
print(cent_df)

# Cluster sizes table
sizes_df = pd.DataFrame({
    'cluster': range(5),
    'size': cluster_counts,
    'percentage': (cluster_counts / len(df) * 100).round(1)
})
sizes_df.to_csv('../outputs/cluster_sizes.csv', index=False)
print("\nSaved ../outputs/cluster_sizes.csv")

### Step 7: 2D & 3D Cluster Visualizations
We plot clusters on Annual Income vs. Spending Score (the primary commercial drivers) with centroids highlighted, and visualize the full 3D space incorporating Age.

In [ ]:
# Archetype Name Mapping based on Centroid Coordinates
segment_names = {
    0: 'Frugal (Low Income, Low Spending)',
    1: 'Impulsive (Young, High Spending)',
    2: 'VIP (High Income, High Spending)',
    3: 'Careful Wealthy (High Income, Low Spending)',
    4: 'Standard (Moderate Income, Moderate Spending)'
}

# 2D Income vs Spending Plot
fig, ax = plt.subplots(figsize=(10, 7), facecolor='#141610')
ax.set_facecolor('#1B1E16')
ax.tick_params(colors='#F2F0E7')
ax.xaxis.label.set_color('#F2F0E7')
ax.yaxis.label.set_color('#F2F0E7')
ax.title.set_color('#F2F0E7')
for spine in ax.spines.values():
    spine.set_color('#333333')

palette = ['#FF6B6B', '#C8FF32', '#78B7FF', '#FFA07A', '#D8B4E2']
for c in range(5):
    cluster_pts = df[df['cluster'] == c]
    ax.scatter(
        cluster_pts['Annual Income (k$)'],
        cluster_pts['Spending Score (1-100)'],
        label=f"Cluster {c}: {segment_names[c].split(' (')[0]} (n={len(cluster_pts)})",
        color=palette[c],
        s=90,
        alpha=0.85,
        edgecolors='#141610',
        linewidth=1
    )

# Centroids
ax.scatter(
    centroids_real[:, 1],
    centroids_real[:, 2],
    marker='X',
    s=320,
    color='#FFFFFF',
    edgecolors='#141610',
    linewidth=2.5,
    label='Cluster Centroids'
)

for c in range(5):
    ax.annotate(
        f"C{c}: {segment_names[c].split(' (')[0]}\n(${centroids_real[c, 1]:.0f}k, {centroids_real[c, 2]:.0f})",
        (centroids_real[c, 1], centroids_real[c, 2]),
        xytext=(0, 16),
        textcoords='offset points',
        ha='center',
        fontsize=9,
        fontweight='bold',
        color='#F2F0E7',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#22261C', edgecolor='#444444', alpha=0.9)
    )

ax.set_xlabel('Annual Income (k$)', fontsize=13, labelpad=10)
ax.set_ylabel('Spending Score (1-100)', fontsize=13, labelpad=10)
ax.set_title('Customer Segmentation — Income vs Spending (k=5 Clusters)', fontsize=14, weight='bold', pad=15)
ax.legend(facecolor='#22261C', edgecolor='#333333', labelcolor='#F2F0E7', loc='upper right')
ax.grid(True, linestyle=':', alpha=0.25, color='#F2F0E7')
plt.tight_layout()
plt.savefig('../outputs/clusters_income_spending.png', dpi=150, bbox_inches='tight')
plt.show()

# 3D Plot: Age x Income x Spending
fig = plt.figure(figsize=(11, 8), facecolor='#141610')
ax3 = fig.add_subplot(111, projection='3d', facecolor='#141610')
ax3.xaxis.set_pane_color((0.11, 0.12, 0.09, 1.0))
ax3.yaxis.set_pane_color((0.11, 0.12, 0.09, 1.0))
ax3.zaxis.set_pane_color((0.11, 0.12, 0.09, 1.0))
ax3.tick_params(colors='#F2F0E7')
ax3.set_xlabel('Age (Years)', color='#F2F0E7', fontsize=11, labelpad=8)
ax3.set_ylabel('Annual Income (k$)', color='#F2F0E7', fontsize=11, labelpad=8)
ax3.set_zlabel('Spending Score (1-100)', color='#F2F0E7', fontsize=11, labelpad=8)
ax3.set_title('3D Customer Space: Age × Income × Spending (k=5)', color='#F2F0E7', fontsize=14, weight='bold', pad=20)

for c in range(5):
    pts = df[df['cluster'] == c]
    ax3.scatter(
        pts['Age'], pts['Annual Income (k$)'], pts['Spending Score (1-100)'],
        color=palette[c], s=60, alpha=0.8, edgecolors='#141610',
        label=f"C{c}: {segment_names[c].split(' (')[0]}"
    )

ax3.scatter(
    centroids_real[:, 0], centroids_real[:, 1], centroids_real[:, 2],
    marker='X', s=250, color='#FFFFFF', edgecolors='#141610', linewidth=2, label='Centroids'
)
ax3.legend(facecolor='#22261C', edgecolor='#333333', labelcolor='#F2F0E7', loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/clusters_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualizations saved to outputs/.")

### Step 8: Behavioral Profiling & Customer Archetypes
We aggregate each cluster's mean age, income, spending, and female proportion to formulate actionable commercial segments.

In [ ]:
profile = df.groupby('cluster').agg(
    size=('Age', 'size'),
    avg_age=('Age', 'mean'),
    std_age=('Age', 'std'),
    avg_income=('Annual Income (k$)', 'mean'),
    avg_spending=('Spending Score (1-100)', 'mean'),
    pct_female=('Gender', lambda s: (s == 'Female').mean() * 100)
).round(2)

profile['segment_name'] = [segment_names[c] for c in profile.index]
profile.to_csv('../outputs/cluster_profiles.csv')
print("Complete Cluster Profiles (outputs/cluster_profiles.csv):")
print(profile)

# Save serialized models for production inference
joblib.dump(km, '../models/kmeans_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
print("\nSaved models/kmeans_model.joblib and models/scaler.joblib")

### Step 9: Actionable Business Recommendations

Based on the empirical centroids, we provide marketing strategies tailored to each archetype:

1. **Careful Wealthy (Cluster 3: Income ~$86k, Spending ~19, Age ~40) — Highest-ROI Acquisition Target:**  
   These individuals possess substantial purchasing power but exhibit minimal mall expenditure. Rather than standard discount promotions, marketing should deploy high-touch concierge services, private brand previews, and premium loyalty perks. Converting even 10% of this segment into higher spending tiers yields disproportionate margin expansion.

2. **VIP (Cluster 2: Income ~$86k, Spending ~82, Age ~33) — Retention Over Acquisition:**  
   This younger, affluent segment drives the retail center's high-ticket gross merchandise value (GMV). Losing a single VIP customer costs substantially more than onboarding three standard shoppers. Retain them through exclusive product allocations, invitation-only VIP lounges, and fast-track loyalty points. Closely monitor churn indicators.

3. **Impulsive (Cluster 1: Income ~$41k, Spending ~62, Age ~25) — Ethical Engagement & Digital Micro-Targeting:**  
   This youthful cohort spends aggressively despite modest earnings. Target with trend-driven fashion, experiential pop-ups, and mobile social commerce. Ethically cap aggressive credit promotions and high-interest financing offers to preserve long-term brand goodwill and financial responsibility.

---

### 🎓 5 Essential Clustering Interview Questions to Master:
1. **Why is scaling mandatory for K-Means?** K-Means relies on Euclidean distance; unscaled high-variance features dominate distance geometry.
2. **What does a Silhouette Score of 0.42 mean?** Points have reasonable separation with good internal cohesion ($>0$ indicates good clustering; $1$ is perfect).
3. **Why use `k-means++`?** It spaces out initial centroid seeds proportionally to distance squared, avoiding poor local minima.
4. **How does K-Means handle outliers?** Outliers pull centroids away from dense distributions because K-Means minimizes squared Euclidean errors.
5. **Why can't K-Means discover arbitrary shapes?** K-Means creates linear Voronoi boundaries, assuming spherical, convex clusters.